<a href="https://colab.research.google.com/github/DrKroll/llm-humanidades-salamanca/blob/main/notebooks/dia1_python/P3_leer_y_analizar_el_quijote.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# P3 · Leer y analizar el *Quijote*

⏱️ **≈ 105 min** (lunes 13:15–14:00 la parte A, 15:00–16:00 la parte B)

| Parte | Contenido | Tiempo |
|---|---|---|
| A | Ficheros: descargar el *Quijote* de Gutenberg (con copia local), leer, limpiar, guardar | ≈ 45 min |
| B | Análisis: frecuencias, gráficos, palabra clave en contexto (KWIC) y un primer «modelo de lenguaje» con bigramas | ≈ 60 min |

In [ ]:
#@title ⚙️ Preparación: ejecuta esta celda primero (no hace falta leer el código) { display-mode: "form" }
# ⚙️ Preparación: ejecuta esta celda primero (Mayús + Intro).
# Descarga los textos de ejemplo del curso (solo hace falta una vez).
import os, sys
REPO_URL = "https://github.com/DrKroll/llm-humanidades-salamanca"
REPO_DIR = "llm-humanidades-salamanca"
if "google.colab" in sys.modules:
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL}
    os.chdir(REPO_DIR)
else:
    while not os.path.exists("config.yaml") and os.getcwd() != "/":
        os.chdir("..")
print("Carpeta de trabajo:", os.getcwd())

# Parte A · Ficheros: leer el *Quijote* entero

**Objetivo:** leer un archivo de texto, entender la codificación (UTF-8) y descargar un libro de Project Gutenberg.

Usaremos *Don Quijote* (Project Gutenberg, libro n.º 2000, dominio público). Si la descarga falla
(Gutenberg a veces no responde), usamos la **copia local** que hay en el repositorio.

## 1. Descargar de Project Gutenberg (con plan B)

In [ ]:
import urllib.request
from pathlib import Path

URL = "https://www.gutenberg.org/cache/epub/2000/pg2000.txt"
COPIA_LOCAL = Path("data/gutenberg/pg2000_don_quijote.txt")

try:
    with urllib.request.urlopen(URL, timeout=15) as respuesta:
        texto = respuesta.read().decode("utf-8")
    if "Quijote" not in texto:
        raise ValueError("La página descargada no es el libro")
    print("Descargado de Gutenberg ✅")
except Exception as e:
    print("No se pudo descargar (", e, ") → usamos la copia local")
    texto = COPIA_LOCAL.read_text(encoding="utf-8")

print(f"{len(texto):,} caracteres".replace(",", "."))

## 2. Leer un archivo con `open`
Así se lee cualquier archivo `.txt`. `encoding="utf-8"` evita problemas con los acentos.

In [ ]:
with open(COPIA_LOCAL, encoding="utf-8") as f:
    texto_local = f.read()
print(texto_local[:300])

## 3. Quitar la cabecera y el pie de Gutenberg
Los libros de Gutenberg llevan avisos en inglés al principio y al final. Los buscamos con `find`.

In [ ]:
inicio = texto.find("*** START OF")
inicio = texto.find("\n", inicio) + 1          # saltamos esa línea
fin = texto.find("*** END OF")
libro = texto[inicio:fin].strip() if fin != -1 else texto[inicio:].strip()
print(libro[:500])
print("...")
print(libro[-300:])

## 4. Dividir en partes y capítulos

In [ ]:
corte = libro.find("Segunda parte del ingenioso caballero", 20000)
parte1, parte2 = libro[:corte], libro[corte:]
print(f"Parte I: {len(parte1.split()):,} palabras".replace(",", "."))
print(f"Parte II: {len(parte2.split()):,} palabras".replace(",", "."))
print("Capítulos (aprox.):", libro.count("Capítulo"))

## 5. Guardar un archivo

In [ ]:
Path("outputs").mkdir(exist_ok=True)
with open("outputs/quijote_limpio.txt", "w", encoding="utf-8") as f:
    f.write(libro)
print("Guardado en outputs/quijote_limpio.txt")

## 🖐️ Tu turno
1. Lee el archivo `data/sample/lazarillo_de_tormes.txt` y cuenta sus palabras.
2. ¿Cuántas veces aparece «Sancho» en la Parte I y en la Parte II?

In [ ]:
# Escribe aquí tu código

<details><summary>💡 Ver una solución</summary>

```python
laz = Path("data/sample/lazarillo_de_tormes.txt").read_text(encoding="utf-8")
print(len(laz.split()))
print(parte1.count("Sancho"), parte2.count("Sancho"))
```
</details>

# Parte B · Análisis sencillo de texto: frecuencias, KWIC y bigramas

**Objetivo:** contar palabras, buscar en contexto, hacer gráficos... y construir un primer «modelo de lenguaje» con bigramas.

## 1. Cargar y limpiar
> 🔁 ¿Se ha desconectado Colab durante la pausa? Ejecuta primero la celda ⚙️ del principio del cuaderno. Esta celda vuelve a cargar el texto.

In [ ]:
import re
from pathlib import Path
texto = Path("data/gutenberg/pg2000_don_quijote.txt").read_text(encoding="utf-8")
texto = texto[texto.find("\n", texto.find("*** START OF")) + 1 : texto.find("*** END OF")]

def tokenizar(texto):
    """Minúsculas y solo letras (incluye acentos y ñ)."""
    return re.findall(r"[a-záéíóúüñ]+", texto.lower())

palabras = tokenizar(texto)
print(len(palabras), "palabras;", len(set(palabras)), "distintas")
print(palabras[:20])

## 2. Frecuencias con `Counter`

In [ ]:
from collections import Counter
frecuencias = Counter(palabras)
frecuencias.most_common(15)

## 3. Quitar palabras vacías (*stopwords*)

In [ ]:
VACIAS = set("""de la que el en y a los se del las un por con no una su para es al lo como
más mas pero sus le ya o este sí si porque esta entre cuando muy sin sobre también me hasta hay donde
quien desde todo nos ni ese eso yo él mi tan así había era fue dijo había lo le les me te tu vuestra
merced os ha he está estaba ser sino pues aunque cual otro otra esto qué todos todas""".split())
contenido = Counter(p for p in palabras if p not in VACIAS and len(p) > 2)
contenido.most_common(20)

## 4. Gráfico de barras

In [ ]:
import matplotlib.pyplot as plt
top = contenido.most_common(20)
plt.figure(figsize=(10, 4))
plt.bar([p for p, n in top], [n for p, n in top], color="#8c2d19")
plt.xticks(rotation=60, ha="right")
plt.title("Palabras más frecuentes del Quijote (sin palabras vacías)")
plt.tight_layout()
plt.show()

## 5. Concordancia (KWIC: *keyword in context*)

In [ ]:
def kwic(texto, palabra, ancho=40, maximo=10):
    plano = texto.replace("\n", " ")
    for i, m in enumerate(re.finditer(rf"\b{palabra}\b", plano, flags=re.IGNORECASE)):
        if i >= maximo:
            break
        izq = plano[max(0, m.start() - ancho): m.start()]
        der = plano[m.end(): m.end() + ancho]
        print(f"{izq:>{ancho}} [{m.group()}] {der}")

kwic(texto, "molinos")

## 6. Comparar la Parte I y la Parte II

In [ ]:
corte = texto.find("Segunda parte del ingenioso caballero", 20000)
p1, p2 = Counter(tokenizar(texto[:corte])), Counter(tokenizar(texto[corte:]))
n1, n2 = sum(p1.values()), sum(p2.values())
for w in ["dulcinea", "sancho", "ínsula", "duque", "encantadores", "rocinante"]:
    print(f"{w:>12}: Parte I {1000 * p1[w] / n1:5.2f} ‰   Parte II {1000 * p2[w] / n2:5.2f} ‰")

## 7. Del recuento al modelo de lenguaje: bigramas
Un **modelo de lenguaje** predice la siguiente palabra. El más simple: contar qué palabra sigue a cada palabra
(**bigramas**) y elegir al azar según esas frecuencias. ¡Esto es lo que hacen los LLM, pero con mucho más contexto!

In [ ]:
import random
from collections import defaultdict

siguientes = defaultdict(Counter)
for a, b in zip(palabras, palabras[1:]):
    siguientes[a][b] += 1

print("Después de 'don':", siguientes["don"].most_common(5))
print("Después de 'sancho':", siguientes["sancho"].most_common(5))

In [ ]:
def generar(inicio="don", n=25, semilla=None):
    random.seed(semilla)
    palabra, salida = inicio, [inicio]
    for _ in range(n):
        opciones = siguientes.get(palabra)
        if not opciones:
            break
        palabra = random.choices(list(opciones), weights=opciones.values())[0]
        salida.append(palabra)
    return " ".join(salida)

for s in range(3):
    print(generar("sancho", semilla=s))

## 🖐️ Tu turno
1. Busca en contexto (KWIC) «Dulcinea», «libros» o «gigantes».
2. Haz el gráfico de las palabras más frecuentes de la **Parte II**.
3. Genera texto con bigramas empezando por otras palabras. ¿Por qué pierde el sentido tan rápido?
4. **Reto:** haz un modelo de **trigramas** (las dos palabras anteriores predicen la siguiente).
   ¿Mejora? ¿Copia frases enteras del original? 👉 Esto lo retomamos en la clase sobre LLM.

In [ ]:
# Escribe aquí tu código

<details><summary>💡 Ver una solución</summary>

```python
trigramas = defaultdict(Counter)
for a, b, c in zip(palabras, palabras[1:], palabras[2:]):
    trigramas[(a, b)][c] += 1

def generar3(a="dijo", b="don", n=25, semilla=0):
    random.seed(semilla)
    salida = [a, b]
    for _ in range(n):
        opciones = trigramas.get((a, b))
        if not opciones:
            break
        c = random.choices(list(opciones), weights=opciones.values())[0]
        salida.append(c)
        a, b = b, c
    return " ".join(salida)

print(generar3())
```
</details>